# Alternativas a OpenAI — Gemini, Groq, Cerebras y Ollama Cloud

_APIs frontera de Google y proveedores ultra-rápidos de modelos open-source_

**Módulo 3 — Introducción a AI Engineering** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![AI Engineering](assets/header.png)

## 1. ¿Por qué considerar alternativas a OpenAI?

OpenAI es el _default_ pero rara vez es la única opción razonable. Razones para mirar alternativas:

| Razón | Explicación |
|---|---|
| **Privacidad / regulación** | Tus datos (médicos, legales, financieros) no pueden salir de tu infraestructura. Alternativas serias incluyen self-hosting con vLLM/TGI o cloud regional. |
| **Costo** | Volumen alto + tarea simple → un modelo open-source vía Groq/Cerebras o un Haiku/Flash sale 10-100× más barato. |
| **Latencia** | Inferencia en hardware especializado (Groq LPU, Cerebras WSE) = 5-10× más rápida que GPU típica. |
| **Vendor lock-in** | Diversificar proveedores te protege de cambios de pricing, deprecaciones de modelos, downtime. |
| **Customización** | Open-source te deja fine-tunear, cuantizar, modificar la arquitectura. |
| **Frontera** | A veces _no_ es OpenAI quien tiene el mejor modelo para tu tarea (Claude para código, Gemini para multimodal de contexto largo). |

En este notebook cubrimos cuatro alternativas representativas:
- **Gemini** — la familia de modelos de Google, con context window enorme (1-2M tokens) y multimodal nativo.
- **Groq** — inferencia ultra-rápida de modelos open-source sobre LPUs.
- **Cerebras** — inferencia ultra-rápida sobre Wafer-Scale Engine, throughput líder en modelos grandes.
- **Ollama Cloud** — modelos open-source MUY grandes (DeepSeek 671B, Qwen3 480B, Kimi K2 1T) sin necesidad de GPU local.

## 2. Cargar credenciales desde `ai.env`

Igual que en el notebook 02, usamos un archivo `ai.env` (ya está en `.gitignore`) junto al notebook con todas las keys que vamos a usar:

```
# modulo-3-introduccion-ai-engineering/ai.env
OPENAI_API_KEY=sk-...        # opcional aquí (truco "Ollama con SDK de OpenAI")
GOOGLE_API_KEY=...           # https://aistudio.google.com/app/apikey
GROQ_API_KEY=gsk_...         # https://console.groq.com/keys
CEREBRAS_API_KEY=csk-...     # https://cloud.cerebras.ai/
OLLAMA_API_KEY=...           # https://ollama.com/settings/keys  (Ollama Cloud)
HF_TOKEN=hf_...              # opcional, no se usa en este notebook
```

Cada proveedor te da su key en su propia consola. Ninguno necesita tarjeta para el tier gratuito.

In [1]:
# Cargar credenciales desde ai.env (mismo patrón que el notebook 02)
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('ai.env') if Path('ai.env').exists() else Path('.env')
load_dotenv(env_path)

def _mask(v):
    return f'{v[:6]}…{v[-4:]}' if v and len(v) > 12 else '***'

for k in ('OPENAI_API_KEY', 'GOOGLE_API_KEY', 'GROQ_API_KEY', 'CEREBRAS_API_KEY', 'OLLAMA_API_KEY', 'HF_TOKEN'):
    v = os.environ.get(k)
    flag = '✅' if v else 'ℹ️ '
    print(f'{flag} {k:<18} ({_mask(v) if v else "no configurada"})')

print(f'\nFuente: {env_path}')

✅ OPENAI_API_KEY     (sk-pro…MC0A)
✅ GOOGLE_API_KEY     (AIzaSy…4s-w)
✅ GROQ_API_KEY       (gsk_cz…f1j3)
✅ CEREBRAS_API_KEY   (csk-x6…y8yj)
✅ OLLAMA_API_KEY     (204174…HFeg)
✅ HF_TOKEN           (hf_dWD…MfBV)

Fuente: ai.env


## 3. El ecosistema de modelos open-source

Una muestra de los modelos open-source más usados en 2025-2026:

| Modelo | Tamaños | Fuerte en | Licencia |
|---|---|---|---|
| **Llama 3.1 / 3.3** (Meta) | 8B / 70B / 405B | propósito general, instrucciones | Llama Community License |
| **Mistral / Mixtral** | 7B / 8x7B / 8x22B | eficiencia, MoE | Apache 2.0 (algunos) |
| **Qwen 2.5** (Alibaba) | 0.5B – 72B | multilingüe, código | Apache 2.0 |
| **Gemma 2** (Google) | 2B / 9B / 27B | tamaños pequeños eficientes | Gemma License |
| **Phi-3 / Phi-4** (Microsoft) | 3.8B – 14B | razonamiento por su tamaño | MIT |
| **DeepSeek V3 / R1** | 67B+ | razonamiento (R1), código (V3) | MIT |

> "Open-source" en LLMs es un espectro: algunos liberan **pesos + código + datos**, la mayoría solo **pesos** con licencia que limita uso comercial.

## 4. Ollama Cloud — modelos open-source grandes desde la nube

[Ollama](https://ollama.com/) es conocido por ser el "Docker para LLMs" — un runtime para correr Llama, Mistral, Qwen, Gemma, etc. Localmente funciona genial, pero correr modelos grandes (>70B) requiere hardware caro.

**Ollama Cloud** resuelve eso: la **misma CLI y misma librería** que usarías en local funcionan contra modelos servidos desde la infraestructura de Ollama, incluyendo modelos demasiado grandes para tu Mac/PC (DeepSeek 671B, Qwen3 480B, GPT-OSS 120B, Kimi K2 1T).

**Características clave:**
- API HTTP **compatible con OpenAI** — puedes apuntar el SDK de OpenAI a Ollama cambiando solo `base_url`.
- **Misma interfaz** que Ollama local: si mañana quisieras moverte a self-hosting, el código no cambia.
- Tier gratuito + planes pagos. Sin tarjeta para empezar.
- **No necesitas GPU local** — ideal para talleres, prototipado, demos.

> 💡 En este curso usamos **solo el modo cloud** para simplificar el setup. Si te interesa correr modelos en tu máquina, instala [Ollama](https://ollama.com/download) localmente: la API es idéntica, solo cambia el `host` del cliente.

### 4.1 Setup

```bash
# 1. Crear key gratis en https://ollama.com/settings/keys
# 2. Agregar a ai.env:  OLLAMA_API_KEY=...
# 3. Instalar la librería oficial
uv add ollama
```

La librería de Python es la misma para local y cloud — solo cambia el `host` y los headers de Authorization.

In [38]:
# Chat con Ollama Cloud — la librería ollama apuntada a https://ollama.com
# uv add ollama
import os
from ollama import Client

ollama_cloud = Client(
    host='https://ollama.com',
    headers={'Authorization': f"Bearer {os.environ.get('OLLAMA_API_KEY')}"},
)

resp = ollama_cloud.chat(
    model='gpt-oss:120b-cloud',     # sufijo -cloud → Ollama Cloud
    messages=[
        {'role': 'system', 'content': 'Eres un asistente conciso. Responde en máximo 3 frases.'},
        {'role': 'user',   'content': '¿Qué es un Transformer? pero los de las peliculas'},
    ],
    options={
        'temperature': 0.3,
        'num_predict': 200,        # equivalente a max_tokens
    },
)
print(resp['message']['content'])
print('---')
print(f"Tokens evaluados: {resp.get('eval_count', 'n/a')}")

Los Transformers son una raza extraterrestre de robots inteligentes que pueden cambiar su forma física, convirtiéndose en vehículos, armas o animales. Cada uno pertenece a una facción: los heroicos Autobots y los malvados Decepticons, que luchan por el control de la Tierra y del poderoso AllSpark. Su capacidad de transformación combina tecnología avanzada con energía de origen cósmico, lo que les permite adaptarse y combatir en cualquier entorno.
---
Tokens evaluados: 164


In [39]:
# Streaming token a token con Ollama Cloud
stream = ollama_cloud.chat(
    model='qwen3-coder:480b-cloud',
    messages=[{'role': 'user', 'content': 'Escribe una función Python que invierta una lista enlazada.'}],
    stream=True,
)

for chunk in stream:
    print(chunk['message']['content'], end='', flush=True)

Claro, a continuación te muestro una implementación de una **lista enlazada** en Python, junto con una **función para invertirla**.

### Paso 1: Definir el nodo de la lista enlazada

```python
class Nodo:
    def __init__(self, dato):
        self.dato = dato
        self.siguiente = None
```

### Paso 2: Definir la lista enlazada y la función para invertirla

```python
class ListaEnlazada:
    def __init__(self):
        self.cabeza = None

    def agregar(self, dato):
        nuevo_nodo = Nodo(dato)
        if not self.cabeza:
            self.cabeza = nuevo_nodo
        else:
            actual = self.cabeza
            while actual.siguiente:
                actual = actual.siguiente
            actual.siguiente = nuevo_nodo

    def mostrar(self):
        elementos = []
        actual = self.cabeza
        while actual:
            elementos.append(actual.dato)
            actual = actual.siguiente
        return elementos

    def invertir(self):
        previo = None
        act

### 4.2 Modelos disponibles en Ollama Cloud

Algunos modelos servidos en cloud (catálogo cambia rápido — ver https://ollama.com/library):

| Modelo | Notas |
|---|---|
| `gpt-oss:120b-cloud` / `gpt-oss:20b-cloud` | reasoning open-source de OpenAI |
| `qwen3-coder:480b-cloud` | código, MoE muy grande |
| `deepseek-v3.1:671b-cloud` | propósito general frontier-class |
| `kimi-k2:1t-cloud` | un trillón de parámetros |
| `glm-4.6:cloud` | propósito general |

### 4.3 Truco: usar el SDK de OpenAI apuntando a Ollama

Como Ollama (local y cloud) expone una API compatible con OpenAI, puedes reusar tu código del SDK de OpenAI cambiando solo `base_url` y `api_key`:

```python
from openai import OpenAI

client = OpenAI(
    base_url='https://ollama.com/v1',
    api_key=os.environ['OLLAMA_API_KEY'],
)

resp = client.chat.completions.create(
    model='gpt-oss:120b-cloud',
    messages=[{'role': 'user', 'content': 'Hola, ¿quién eres?'}],
)
print(resp.choices[0].message.content)
```

Esto te da una **abstracción de proveedor gratis**: la misma forma de llamar funciona contra OpenAI, Groq, Cerebras y Ollama Cloud — solo cambias `base_url`. Es el mismo truco que veremos con Groq y Cerebras.

## 5. Gemini — la familia de modelos de Google

[Gemini](https://ai.google.dev/) es el LLM frontera de Google. Familias principales:

| Modelo | Fuerte en | Context window |
|---|---|---|
| **Gemini 2.5 Pro** | razonamiento profundo, código, multimodal | 2M tokens |
| **Gemini 2.5 Flash** | latencia y costo bajos, alta calidad | 1M tokens |
| **Gemini 2.5 Flash-Lite** | volumen masivo a costo mínimo | 1M tokens |
| **Gemma 3** | open weights (corre en Ollama) | varía |

**Diferenciadores frente a OpenAI:**
- **Context window enorme** (1-2 M tokens) — útil para mandar PDFs gigantes, libros, repos enteros.
- **Multimodal nativo** — texto + imagen + video + audio en la misma llamada.
- **Pricing competitivo** — Flash es de los más baratos del mercado.
- **Tier gratuito generoso** para prototipado vía Google AI Studio.

### 5.1 Setup

```bash
# 1. Crear key gratis en https://aistudio.google.com/app/apikey
# 2. Agregar a ai.env:  GOOGLE_API_KEY=...
# 3. Instalar el SDK NUEVO unificado (reemplaza al deprecado google-generativeai)
uv add google-genai
```

> ⚠️ El paquete antiguo `google-generativeai` está **deprecated** desde finales de 2024. El nuevo SDK unificado es `google-genai` (`from google import genai`) — lo usamos abajo.

La key ya quedó cargada en la sección 2 desde `ai.env`.

In [2]:
# Chat con Gemini — SDK unificado google-genai
# uv add google-genai
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ.get('GOOGLE_API_KEY'))

resp = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents='¿Qué diferencia hay entre RNN y Transformer?',
    config=types.GenerateContentConfig(
        system_instruction='Eres un asistente conciso. Responde en máximo 3 frases.',
        temperature=0.3,
        max_output_tokens=200,
        top_p=0.9,
        top_k=40,
    ),
)
print(resp.text)

Las RNN procesan secuencias de forma secuencial, lo que puede ser lento y tener dificultades con dependencias a largo plazo. Los Transformers, en cambio, utilizan mecanismos de atención para procesar secuencias en paralelo, lo que les permite capturar mejor las relaciones entre elementos distantes. Esto hace que los Transformers sean más eficientes y efectivos para muchas tareas de procesamiento de lenguaje natural.


In [3]:
# Conversación multi-turno con Gemini
chat = client.chats.create(model='gemini-2.5-flash')

print('USER:', '¿Qué es un embedding?')
print('BOT :', chat.send_message('¿Qué es un embedding?').text)
print('---')
print('USER:', 'Dame un caso de uso concreto.')
print('BOT :', chat.send_message('Dame un caso de uso concreto.').text)
print('---')
print(f'Mensajes en el historial: {len(chat.get_history())}')

USER: ¿Qué es un embedding?
BOT : Un **embedding** (o incrustación, en español) es una representación numérica de un objeto (como una palabra, una imagen, un usuario, un documento completo, o cualquier otra entidad) en un espacio de dimensiones múltiples, donde la *distancia* y la *dirección* entre los objetos en ese espacio capturan sus **similitudes semánticas y relaciones contextuales**.

Para entenderlo mejor, desglosémoslo:

1.  **Representación Numérica (Vector):**
    *   Los ordenadores no entienden palabras, imágenes o conceptos directamente. Necesitan números. Un embedding convierte estos objetos en una **lista de números** (lo que se conoce como un "vector").
    *   Por ejemplo, la palabra "gato" podría ser representada por un vector como `[0.2, -0.5, 0.8, 1.3, ...]`, y la palabra "perro" por `[0.3, -0.4, 0.7, 1.2, ...]`. Estos números son generados por modelos de aprendizaje automático.

2.  **Espacio de Alta Dimensión:**
    *   Cada número en el vector representa una "di

In [5]:
# Embeddings con Gemini
textos = [
    'Me encanta programar en Python.',
    'Adoro escribir código en Python.',
    'El pan está caliente y crujiente.',
    'La pizza italiana es deliciosa.',
]

resp = client.models.embed_content(
    model='gemini-embedding-2',
    contents=textos,
    config=types.EmbedContentConfig(task_type='SEMANTIC_SIMILARITY'),
)

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
embs = np.array([e.values for e in resp.embeddings])
print('Shape:', embs.shape)
print('\nSimilitud coseno:')
print(cosine_similarity(embs).round(2))

Shape: (1, 3072)

Similitud coseno:
[[1.]]


In [9]:
# Multimodal nativo: pasarle una imagen + texto
import PIL.Image
import requests
from io import BytesIO

# Bajamos una imagen pública
url = 'assets/transformer_architecture.png'
img = PIL.Image.open(url)

resp = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=['Describe esta imagen en una sola frase.', img],
)
print(resp.text)

Este diagrama ilustra la arquitectura de una red neuronal Transformer, detallando sus bloques codificador y decodificador con autoatención multi-cabeza (enmascarada en el decodificador), atención cruzada y redes feed-forward.


### 5.2 Streaming con Gemini

```python
stream = client.models.generate_content_stream(
    model='gemini-2.5-flash',
    contents='Cuenta una historia corta',
)
for chunk in stream:
    if chunk.text:
        print(chunk.text, end='', flush=True)
```

### 5.3 Function calling

Gemini también soporta tool use con una API parecida a OpenAI:

```python
def get_weather(city: str) -> dict:
    """Devuelve el clima para una ciudad."""
    return {'city': city, 'temp_c': 22, 'desc': 'soleado'}

resp = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='¿Qué tiempo hace en Lima?',
    config=types.GenerateContentConfig(tools=[get_weather]),
)
print(resp.text)
```

## 6. Groq — inferencia ultra-rápida con LPUs

[Groq](https://groq.com/) sirve modelos open-source corriendo sobre hardware **LPU (Language Processing Unit)** — chips diseñados específicamente para inferencia de LLMs. Resultado: latencias y throughput **muy superiores** a GPUs convencionales.

| | Groq (LPU) | GPU típica en cloud |
|---|---|---|
| Throughput | **300-800 tokens/s** | 30-100 tokens/s |
| Latencia primer token | ~50 ms | 200-500 ms |
| Modelos | Llama 3.1/3.3, Mixtral, Gemma, Qwen, DeepSeek R1 | depende del proveedor |

**Por qué importa:** en aplicaciones interactivas (chat, agentes, voz), la velocidad cambia la sensación del producto. Una respuesta que tarda 500 ms vs 5 s es la diferencia entre "instantáneo" y "esperando".

**Otras ventajas:**
- **Tier gratuito generoso** — 30 RPM en la mayoría de modelos, sin tarjeta.
- **API compatible con OpenAI** — solo cambias `base_url` y `api_key`. Mismo truco que con Ollama.
- **Modelos open-source** — no estás casado con un modelo propietario; si Groq mañana sube precios, te llevas el código y lo corres en otro proveedor con los mismos pesos.

### 6.1 Setup

```bash
# 1. Crear key gratis en https://console.groq.com/keys
# 2. Agregar a ai.env:  GROQ_API_KEY=gsk_...
# 3. El SDK de OpenAI ya está instalado, no hace falta uno nuevo.
```

La key ya quedó cargada en la sección 2 desde `ai.env`.

In [10]:
# Groq usando el SDK de OpenAI — el truco está en base_url
import os, time
from openai import OpenAI

groq = OpenAI(
    base_url='https://api.groq.com/openai/v1',
    api_key=os.environ.get('GROQ_API_KEY'),
)

t0 = time.time()
resp = groq.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[
        {'role': 'system', 'content': 'Eres un asistente conciso. Responde en español, máximo 3 frases.'},
        {'role': 'user',   'content': 'Resume qué es Groq y por qué importa.'},
    ],
    temperature=0.3,
    max_tokens=150,
)
elapsed = time.time() - t0

print(resp.choices[0].message.content)
print('---')
print(f'Tokens output: {resp.usage.completion_tokens}')
print(f'Tiempo total : {elapsed:.2f}s  →  {resp.usage.completion_tokens/elapsed:.0f} tok/s')

Groq es una empresa de inteligencia artificial que desarrolla procesadores de aprendizaje automático para acelerar el rendimiento de los modelos de IA. Su tecnología permite un procesamiento más rápido y eficiente de grandes cantidades de datos, lo que es crucial para aplicaciones como el reconocimiento de imágenes y el procesamiento del lenguaje natural. Esto importa porque puede revolucionar la forma en que se desarrollan y se implementan soluciones de IA en diversas industrias.
---
Tokens output: 104
Tiempo total : 1.21s  →  86 tok/s


### 6.2 Streaming con Groq

Aquí es donde se nota la velocidad — el texto aparece tan rápido que apenas alcanzas a leerlo. Útil para experiencias tipo "instantáneo".

In [11]:
# Streaming con Groq — fíjate en la velocidad de tokens/segundo
import time

t0 = time.time()
first_token_at = None
total_tokens = 0

stream = groq.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=[{'role': 'user', 'content': 'Cuenta del 1 al 10 con una palabra creativa por número.'}],
    stream=True,
    stream_options={'include_usage': True},   # último chunk trae usage real
)

for chunk in stream:
    if chunk.usage:
        total_tokens = chunk.usage.completion_tokens
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta.content
    if delta:
        if first_token_at is None:
            first_token_at = time.time() - t0
        print(delta, end='', flush=True)

elapsed = time.time() - t0
print(f'\n\n---')
print(f'Time to first token : {first_token_at*1000:.0f} ms')
print(f'Tiempo total        : {elapsed:.2f} s')
print(f'Tokens generados    : {total_tokens}')
print(f'Throughput          : {total_tokens/elapsed:.0f} tok/s')

¡Claro! Aquí te dejo la cuenta del 1 al 10 con una palabra creativa por número:

1. **Mágico**: Empezamos con un número lleno de sorpresas y encanto.
2. **Veloz**: Aceleramos con este número, lleno de velocidad y emoción.
3. **Tríptico**: Un número que nos hace pensar en tres cosas increíbles.
4. **Fábula**: Un número que nos transporta a un mundo de historias y leyendas.
5. **Dinámico**: Un número lleno de energía y movimiento.
6. **Sinfónico**: Un número que nos hace pensar en melodías y armonías.
7. **Misterioso**: Un número que nos rodea de enigmas y secretos.
8. **Octágono**: Un número que nos hace pensar en formas y estructuras geométricas.
9. **Nebuloso**: Un número que nos transporta a un mundo de nubes y niebla.
10. **Galáctico**: Un número que nos hace sentir conectados con el universo y todas sus maravillas.

Espero que disfrutes de esta cuenta creativa. ¿Quieres que siga contando?

---
Time to first token : 615 ms
Tiempo total        : 1.46 s
Tokens generados    : 270
Throu

### 6.3 Modelos disponibles en Groq

| Modelo | Tokens / s aprox. | Notas |
|---|---|---|
| `llama-3.3-70b-versatile` | ~280 | propósito general, calidad alta |
| `llama-3.1-8b-instant` | ~750 | muy rápido para tareas simples |
| `mixtral-8x7b-32768` | ~500 | MoE, contexto 32K |
| `gemma2-9b-it` | ~500 | versión de Gemma instruction-tuned |
| `deepseek-r1-distill-llama-70b` | ~250 | reasoning, devuelve `<think>...</think>` |
| `qwen-2.5-32b` | ~200 | multilingüe, código |

Catálogo completo y precios: https://console.groq.com/docs/models

> 💡 **Cuándo usar Groq**: aplicaciones donde la latencia importa (chat en tiempo real, voz, agentes con muchos pasos), prototipado sin gastar créditos de OpenAI, fallback de un proveedor caído. **Cuándo NO**: cuando necesitas un modelo propietario específico (GPT-4o, Claude Opus) — Groq solo sirve open-source.

## 7. Cerebras — el otro chip especializado

[Cerebras](https://cloud.cerebras.ai/) compite directamente con Groq en el espacio de **inferencia ultra-rápida**, pero con una arquitectura diferente: el **Wafer-Scale Engine (WSE)**, un chip del tamaño de un plato (no exagero — es literalmente una oblea entera de silicio sin cortar). El resultado es uno de los aceleradores con más memoria on-chip y bandwidth del mercado.

| | Cerebras | Groq | GPU típica |
|---|---|---|---|
| Throughput | **~2,000-3,000 tok/s** (Llama 3.1 70B) | ~280-800 tok/s | 30-100 tok/s |
| Latencia primer token | ~100 ms | ~50 ms | 200-500 ms |
| Modelos | Llama 3.1/3.3, Qwen, DeepSeek R1 | Llama, Mixtral, Gemma, R1 | depende |

**Características:**
- **Velocidades absurdas** — Llama 3.1 70B sirviendo a >2,000 tok/s es algo que solo Cerebras tiene en producción.
- **API compatible con OpenAI** — mismo truco de `base_url`.
- **Tier gratuito** — 1M tokens/día gratis en el plan free, sin tarjeta.
- **SDK oficial** disponible: `cerebras-cloud-sdk` (también puedes usar el SDK de OpenAI directamente).

### 7.1 Setup

```bash
# 1. Crear key gratis en https://cloud.cerebras.ai/
# 2. Agregar a ai.env:  CEREBRAS_API_KEY=csk-...
# 3. Instalar el SDK oficial (opcional — el de OpenAI también funciona)
uv add cerebras-cloud-sdk
```

In [12]:
# Cerebras con su SDK oficial
# uv add cerebras-cloud-sdk
import os, time
from cerebras.cloud.sdk import Cerebras

cerebras = Cerebras(api_key=os.environ.get('CEREBRAS_API_KEY'))

t0 = time.time()
resp = cerebras.chat.completions.create(
    model='llama3.1-8b',
    messages=[
        {'role': 'system', 'content': 'Eres un asistente conciso. Responde en español, máximo 3 frases.'},
        {'role': 'user',   'content': 'Resume qué es Cerebras y por qué es relevante.'},
    ],
    temperature=0.3,
    max_tokens=150,
)
elapsed = time.time() - t0

print(resp.choices[0].message.content)
print('---')
print(f'Tokens output: {resp.usage.completion_tokens}')
print(f'Tiempo total : {elapsed:.2f}s  →  {resp.usage.completion_tokens/elapsed:.0f} tok/s')

Cerebras es una empresa de tecnología que desarrolla un procesador de computadora llamado Wafer Scale Engine (WSE), diseñado para procesar grandes cantidades de datos de manera rápida y eficiente. Este procesador es relevante porque tiene la capacidad de procesar 1 exaFLOPS (1 billón de operaciones por segundo), lo que lo convierte en el procesador más rápido del mundo.
---
Tokens output: 91
Tiempo total : 0.27s  →  337 tok/s


In [13]:
# Cerebras también es compatible con el SDK de OpenAI — mismo truco que Groq/Ollama
from openai import OpenAI

cerebras_oai = OpenAI(
    base_url='https://api.cerebras.ai/v1',
    api_key=os.environ.get('CEREBRAS_API_KEY'),
)

resp = cerebras_oai.chat.completions.create(
    model='llama3.1-8b',
    messages=[{'role': 'user', 'content': 'En una frase: ¿qué es un wafer-scale chip?'}],
    max_tokens=80,
)
print(resp.choices[0].message.content)

Un wafer-scale chip es un tipo de procesador que combina múltiples procesadores o lógica de circuitos en una sola placa, o "wafer", para maximizar la ocupación de espacio y reducir el consumo de energía.


### 7.2 Modelos disponibles en Cerebras

| Modelo | Tokens / s aprox. | Notas |
|---|---|---|
| `llama-3.3-70b` | ~2,200 | propósito general, calidad alta |
| `llama-3.1-8b` | ~2,500 | rapidísimo para tareas simples |
| `llama-4-scout-17b-16e-instruct` | ~2,600 | familia Llama 4 |
| `qwen-3-32b` | ~2,000 | multilingüe, código |
| `deepseek-r1-distill-llama-70b` | ~1,500 | reasoning |

Catálogo completo: https://inference-docs.cerebras.ai/introduction

> 💡 **Cuándo usar Cerebras vs Groq**: ambos son extremadamente rápidos. Cerebras suele ganar en **throughput puro** (tokens/segundo en respuestas largas), Groq en **time-to-first-token** y en variedad de modelos. Para producción seria, conviene tener ambos como fallback uno del otro.

## 8. Comparación entre proveedores

| | OpenAI | Anthropic | Gemini | **Groq** | **Cerebras** | **Ollama Cloud** |
|---|---|---|---|---|---|---|
| **Modelos** | GPT-4o, GPT-5, o3 | Claude Opus / Sonnet / Haiku 4.x | Gemini 2.5 Pro / Flash | Llama, Mixtral, Qwen, R1 (open) | Llama, Qwen, R1 (open) | gpt-oss, Qwen3 480B, DeepSeek 671B, Kimi K2 1T |
| **Pricing** | medio – alto | medio – alto | bajo – medio | **bajo + tier gratis generoso** | **bajo + 1M tok/día gratis** | tier gratuito + planes pagos |
| **Multimodal** | ✅ (vision, audio, TTS) | ✅ (vision, audio) | ✅✅ nativo (incluye video) | parcial (vision en algunos) | parcial | depende del modelo |
| **Context window** | 128K – 1M | 200K – 1M | 1M – 2M | 32K – 128K | 8K – 128K | depende del modelo |
| **Razonamiento** | o1, o3 | thinking en Opus 4 | thinking en 2.5 Pro | DeepSeek R1 | DeepSeek R1 | gpt-oss, DeepSeek |
| **Latencia** | red + inferencia | red + inferencia | red + inferencia | **ultra-rápida (LPU)** | **ultra-rápida (WSE)** | red + inferencia |
| **Lock-in** | alto | alto | alto | bajo (open-source) | bajo (open-source) | bajo (open-source) |
| **API compatible OpenAI SDK** | ✅ (nativa) | ❌ | parcial | ✅ (`base_url` swap) | ✅ (`base_url` swap) | ✅ (`base_url` swap) |
| **Mejor para** | default seguro, ecosistema | código y escritura | contexto largo + multimodal | **velocidad bestial + costo bajo** | **throughput bestial en modelos grandes** | open-source >70B sin GPU local |

## 9. Cuándo elegir cada uno

- **Default razonable para arrancar** → GPT-4o-mini o Claude Sonnet 4. Buena calidad/precio, ecosistema maduro.
- **Tarea frontera (código difícil, razonamiento, escritura)** → Claude Opus 4.x con extended thinking, GPT-5, Gemini 2.5 Pro thinking.
- **Volumen masivo, tarea simple (clasificación, extracción)** → Haiku 4, GPT-4o-mini, Gemini 2.5 Flash-Lite o `llama-3.1-8b-instant` en Groq.
- **Documentos largos / repos enteros** → Gemini 2.5 (1-2M de contexto) o Claude (200K + caching).
- **Latencia mínima (chat tiempo real, voz, agentes multi-paso)** → **Groq** (time-to-first-token ~50 ms) o **Cerebras** (throughput >2,000 tok/s).
- **Probar modelos open-source sin instalar nada** → **Groq** o **Cerebras** te corren Llama 3.3 70B desde una API en segundos, con tier gratuito.
- **Modelos open-source MUY grandes (>70B) sin GPU local** → **Ollama Cloud** (DeepSeek 671B, Qwen3 480B, Kimi K2 1T).
- **Multimodal serio (video, audio largo)** → Gemini 2.5 Pro.
- **Datos que no pueden salir de tu infra** → self-host con vLLM/TGI o Ollama corriendo en tu propio cluster (fuera del scope de este notebook).

## 10. Patrón de abstracción del proveedor

En producción, **no acoples tu lógica a un SDK concreto**. Encapsula las llamadas detrás de una pequeña interfaz para poder cambiar de proveedor sin tocar el resto del código:

```python
class LLMClient:
    def chat(self, messages, **opts) -> str: ...

class OpenAIClient(LLMClient):
    def __init__(self):
        self.client = OpenAI()  # usa OPENAI_API_KEY del entorno
    def chat(self, messages, **opts):
        return self.client.chat.completions.create(
            model='gpt-4o-mini', messages=messages, **opts
        ).choices[0].message.content

class GroqClient(LLMClient):
    def __init__(self):
        self.client = OpenAI(base_url='https://api.groq.com/openai/v1',
                             api_key=os.environ['GROQ_API_KEY'])
    def chat(self, messages, **opts):
        return self.client.chat.completions.create(
            model='llama-3.3-70b-versatile', messages=messages, **opts
        ).choices[0].message.content

class CerebrasClient(LLMClient):
    def __init__(self):
        self.client = OpenAI(base_url='https://api.cerebras.ai/v1',
                             api_key=os.environ['CEREBRAS_API_KEY'])
    def chat(self, messages, **opts):
        return self.client.chat.completions.create(
            model='llama-3.3-70b', messages=messages, **opts
        ).choices[0].message.content

class OllamaCloudClient(LLMClient):
    def __init__(self):
        self.client = OpenAI(base_url='https://ollama.com/v1',
                             api_key=os.environ['OLLAMA_API_KEY'])
    def chat(self, messages, **opts):
        return self.client.chat.completions.create(
            model='gpt-oss:120b-cloud', messages=messages, **opts
        ).choices[0].message.content

class GeminiClient(LLMClient): ...  # usa google-genai (no es compatible OpenAI)
```

Beneficios:
- **A/B testing** de proveedores por feature flag.
- **Fallback** automático si un proveedor cae (ej. si OpenAI 429, te vas a Groq → Cerebras).
- **Tests** con un fake `LLMClient` que devuelve respuestas determinísticas.
- **Migración** entre proveedores sin reescribir lógica de negocio.

> 🎯 Nota: como **Groq, Cerebras y Ollama Cloud** ya exponen API compatible OpenAI, sus `chat()` son básicamente idénticos al de OpenAI — la abstracción te sale casi gratis.

## 11. Cierre del módulo

Lo que tienes ahora:

- Una **base conceptual** de qué son los LLMs y cómo aprendieron lo que saben (notebook 01).
- Un **mapa práctico** del ecosistema de APIs y cómo facturan (notebook 02).
- Las **palancas finas** de prompt + context engineering y entender lo que pasa por debajo (notebook 03).
- **Salidas alternativas** cuando OpenAI no es la mejor opción — Gemini, Groq, Cerebras y Ollama Cloud (este notebook).

A partir de aquí los siguientes pasos naturales son:
- **RAG** — combinar embeddings + un vector store + chat completions.
- **Agentes** — sistemas que usan tools y se planifican en varios pasos.
- **Evals** — cómo medir objetivamente que un cambio mejora.
- **Fine-tuning** — cuando el prompting ya no alcanza.

## 12. Referencias

**Gemini**
- Google AI Studio (donde se generan keys de Gemini) — https://aistudio.google.com/
- Gemini API docs — https://ai.google.dev/gemini-api/docs
- google-genai SDK (nuevo, unificado) — https://github.com/googleapis/python-genai

**Groq**
- Groq — https://groq.com/
- Groq console (API keys) — https://console.groq.com/keys
- Groq model catalog & pricing — https://console.groq.com/docs/models
- Groq + OpenAI SDK compatibility — https://console.groq.com/docs/openai

**Cerebras**
- Cerebras Cloud — https://cloud.cerebras.ai/
- Cerebras Inference docs — https://inference-docs.cerebras.ai/
- cerebras-cloud-sdk (Python) — https://github.com/Cerebras/cerebras-cloud-sdk-python

**Ollama**
- Ollama (local + cloud) — https://ollama.com/
- Ollama Cloud — https://ollama.com/cloud
- Ollama Cloud API keys — https://ollama.com/settings/keys
- Ollama Python client — https://github.com/ollama/ollama-python
- Ollama OpenAI compatibility — https://ollama.com/blog/openai-compatibility

**Ecosistema general**
- Hugging Face — Open LLM Leaderboard: https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard
- Artificial Analysis — comparativa de modelos y pricing en tiempo real: https://artificialanalysis.ai/
- LMSYS Chatbot Arena (ranking por preferencias humanas) — https://lmarena.ai/